In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Utility script for plotting training metrics:

1. Epoch-wise train vs eval loss (single-run logs with 'epoch' + 'train_losstensor(...)', 'eval_losstensor(...)').
2. Boxplot + statistics for pretrain accuracy values.
3. Train vs validation loss for a selected Optuna trial from Train_Eval_Loss.txt.
"""

import re
import os
import numpy as np
import matplotlib.pyplot as plt


# =========================
# 1. EPOCH LOSS FROM TRAIN LOG
# =========================

def plot_epoch_loss(
    log_path: str,
    save_path: str,
    title: str = "Train and Eval Loss over Epochs",
    y_label: str = "Loss Value"
) -> None:
    """
    Parse a log file with lines like:
        epoch0
        train_losstensor(0.1234, ...)
        eval_losstensor(0.2345, ...)
    and plot epoch-wise train vs eval loss.
    """

    epochs = []
    train_loss = []
    eval_loss = []

    train_loss_pattern = r"train_losstensor\(([\d.]+)"
    eval_loss_pattern = r"eval_losstensor\(([\d.]+)"

    with open(log_path, "r") as f:
        for line in f:
            line = line.strip()

            # Detect epoch line
            if line.startswith("epoch"):
                epoch_num = int(re.search(r"\d+", line).group())
                epochs.append(epoch_num)

            # Train loss
            m_train = re.search(train_loss_pattern, line)
            if m_train:
                train_loss.append(float(m_train.group(1)))

            # Eval loss
            m_eval = re.search(eval_loss_pattern, line)
            if m_eval:
                eval_loss.append(float(m_eval.group(1)))

    if not epochs or not train_loss or not eval_loss:
        print(f"[WARN] No valid data parsed from {log_path}")
        return

    # Safety: align lengths
    n = min(len(epochs), len(train_loss), len(eval_loss))
    epochs = epochs[:n]
    train_loss = train_loss[:n]
    eval_loss = eval_loss[:n]

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss, label="Train Loss", marker="o")
    plt.plot(epochs, eval_loss, label="Eval Loss", marker="o")

    plt.xlabel("Training Epoch", fontsize=12, fontweight="bold", labelpad=10)
    plt.ylabel(y_label, fontsize=12, fontweight="bold", labelpad=10)
    plt.title(title, fontweight="bold")

    plt.legend()
    plt.xticks(fontsize=11, fontweight="bold")
    plt.yticks(fontsize=11, fontweight="bold")

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"[OK] Epoch loss plot saved to: {save_path}")


# =========================
# 2. ACCURACY BOXPLOT + STATS
# =========================

def accuracy_boxplot_and_stats(
    acc_file_path: str,
    stats_out_path: str,
    fig_out_path: str,
    title: str = "Accuracy Distribution",
    label: str = "PreTrain Accuracy"
) -> None:
    """
    Read line-wise float accuracies from a file, clean NaNs,
    compute stats, save them, and generate a publication-style boxplot.
    """

    values = []
    with open(acc_file_path, "r") as f:
        for line in f:
            try:
                num = float(line.strip())
                if not np.isnan(num):
                    values.append(num)
            except ValueError:
                continue

    values = [v for v in values if not np.isnan(v)]

    if not values:
        print(f"[WARN] No valid accuracy values found in {acc_file_path}")
        return

    values = np.array(values)

    Q1 = np.percentile(values, 25)
    Q3 = np.percentile(values, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    mean_acc = values.mean()
    median_acc = np.median(values)

    os.makedirs(os.path.dirname(stats_out_path), exist_ok=True)
    with open(stats_out_path, "w") as f:
        f.write(f"Mean accuracy: {mean_acc:.4f}\n")
        f.write(f"Median accuracy: {median_acc:.4f}\n")
        f.write(f"Q1: {Q1:.4f}\nQ3: {Q3:.4f}\nIQR: {IQR:.4f}\n")
        f.write(f"Lower Bound: {lower_bound:.4f}\nUpper Bound: {upper_bound:.4f}\n")

    plt.figure(figsize=(6, 6))
    plt.boxplot(
        values,
        patch_artist=True,
        notch=True,
        boxprops=dict(facecolor="#add8e6", edgecolor="#1f77b4", linewidth=2),
        whiskerprops=dict(color="#1f77b4", linewidth=2),
        capprops=dict(color="#1f77b4", linewidth=2),
        medianprops=dict(color="#d62728", linewidth=2),
        flierprops=dict(
            marker="o",
            markerfacecolor="#555555",
            markeredgecolor="#555555",
            markersize=5,
        ),
    )

    plt.xticks([1], [label], fontsize=12)
    plt.ylabel("Accuracy", fontsize=14, fontweight="bold")
    plt.ylim(0.0, 1.0)
    plt.title(title, fontsize=14, fontweight="bold")
    plt.xticks(fontsize=11, fontweight="bold")
    plt.yticks(fontsize=11, fontweight="bold")

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    os.makedirs(os.path.dirname(fig_out_path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(fig_out_path, dpi=300)
    plt.close()

    print(f"[OK] Accuracy boxplot saved to: {fig_out_path}")
    print(f"[OK] Stats saved to: {stats_out_path}")

